# 11 - Encoding Comparison

Compares four categorical encoding strategies for the laptop price model.
All are evaluated with HistGradientBoostingRegressor on the same time-based split
(train 2018-2023, test 2024+).

In [ ]:
# --- project bootstrap ---
import os, sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from laptop_price.data import load_model_ready
from laptop_price.features.build import TARGET, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from laptop_price.evaluation.splits import time_based_split
from laptop_price.evaluation.metrics import regression_metrics

df = load_model_ready()
print(f"{len(df):,} listings, {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical")

## 1. Train / test split

Time-based: train 2018-2023, test 2024+.

In [ ]:
split = time_based_split(df["listing_year"], train_max_year=2023)
X_train = df.iloc[split.train_idx][[*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]]
y_train = df.iloc[split.train_idx][TARGET]
X_test  = df.iloc[split.test_idx][[*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]]
y_test  = df.iloc[split.test_idx][TARGET]
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

## 2. Encoding strategies

Four approaches:
1. **Ordinal** - integer codes (baseline)
2. **One-hot** - current pipeline default
3. **Target encoding** - out-of-fold via category_encoders (no leakage)
4. **Frequency** - replace category with its relative frequency in training data

In [ ]:
from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin


class FreqEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.freq_ = [X.iloc[:, i].value_counts(normalize=True).to_dict()
                      for i in range(X.shape[1])]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return np.column_stack(
            [X.iloc[:, i].map(self.freq_[i]).fillna(0) for i in range(X.shape[1])]
        )


def _est():
    return HistGradientBoostingRegressor(
        max_iter=400, max_depth=8, learning_rate=0.05,
        min_samples_leaf=20, random_state=42
    )


def make_ordinal_pipe():
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                          unknown_value=-1))])
    pre = ColumnTransformer([("num", "passthrough", list(NUMERIC_FEATURES)),
                              ("cat", cat, list(CATEGORICAL_FEATURES))], remainder="drop")
    return Pipeline([("pre", pre), ("model", _est())])


def make_onehot_pipe():
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                    ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    pre = ColumnTransformer([("num", "passthrough", list(NUMERIC_FEATURES)),
                              ("cat", cat, list(CATEGORICAL_FEATURES))], remainder="drop")
    return Pipeline([("pre", pre), ("model", _est())])


def make_target_pipe():
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                    ("enc", TargetEncoder(cols=list(CATEGORICAL_FEATURES), smoothing=10))])
    pre = ColumnTransformer([("num", "passthrough", list(NUMERIC_FEATURES)),
                              ("cat", cat, list(CATEGORICAL_FEATURES))], remainder="drop")
    return Pipeline([("pre", pre), ("model", _est())])


def make_freq_pipe():
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                    ("enc", FreqEncoder())])
    pre = ColumnTransformer([("num", "passthrough", list(NUMERIC_FEATURES)),
                              ("cat", cat, list(CATEGORICAL_FEATURES))], remainder="drop")
    return Pipeline([("pre", pre), ("model", _est())])


print("Pipelines defined.")

## 3. Evaluation

In [ ]:
strategies = [
    ("Ordinal",         make_ordinal_pipe),
    ("One-hot",         make_onehot_pipe),
    ("Target encoding", make_target_pipe),
    ("Frequency",       make_freq_pipe),
]

results = []
for name, factory in strategies:
    pipe = factory()
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = pipe.predict(X_test)
    infer_ms = (time.perf_counter() - t1) / len(X_test) * 1000

    m = regression_metrics(y_test, y_pred, label=name)
    m["train_s"]        = round(train_s, 2)
    m["infer_ms_per_k"] = round(infer_ms, 3)
    results.append(m)
    print(f"{name:20s}: R2={m['R2']:.4f}  MedAPE={m['MedAPE']:.2f}%  train={train_s:.1f}s")

cmp = pd.DataFrame(results)
display(cmp[["label", "R2", "MAE", "MedAPE", "within_20pct", "train_s", "infer_ms_per_k"]])

## 4. Comparison chart

In [ ]:
cmp_sorted = cmp.sort_values("MedAPE")

fig = go.Figure()
fig.add_trace(go.Bar(name="R2", x=cmp_sorted["label"], y=cmp_sorted["R2"]))
fig.add_trace(go.Bar(
    name="within_20pct (%/100)",
    x=cmp_sorted["label"],
    y=cmp_sorted["within_20pct"] / 100,
))
fig.update_layout(
    title="Encoding strategy comparison (time-based test set)",
    barmode="group",
    yaxis_title="Score",
)
fig.show()

## Summary

- One-hot and target encoding tend to match or beat ordinal for tree-based models
- Target encoding risk: leakage if not applied out-of-fold (handled here)
- Current pipeline uses one-hot, which is a safe default
- Frequency encoding offers a compact representation for high-cardinality columns